In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('../data/kc_house_data.csv')
df.head(5)

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [3]:
# Modifies the original df directly
df.drop(columns=['id'], inplace=True)
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [4]:
# This looks at the 'date' column and converts the text to real dates
df['date'] = pd.to_datetime(df['date'])

In [5]:
# Extract the year and make a new column
df['year_sold'] = df['date'].dt.year

In [6]:
# Calculate how old the house was when it was sold
df['house_age'] = df['year_sold'] - df['yr_built']

In [7]:
# Drop the original text date column
df = df.drop(['date'], axis=1)

In [8]:
# Print just the built, sold, and age columns to verify
print(df[['yr_built', 'year_sold', 'house_age']].head())

   yr_built  year_sold  house_age
0      1955       2014         59
1      1951       2014         63
2      1933       2015         82
3      1965       2014         49
4      1987       2015         28


In [9]:
df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,year_sold,house_age
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,...,0,1955,0,98178,47.5112,-122.257,1340,5650,2014,59
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,...,400,1951,1991,98125,47.7210,-122.319,1690,7639,2014,63
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,...,0,1933,0,98028,47.7379,-122.233,2720,8062,2015,82
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,...,910,1965,0,98136,47.5208,-122.393,1360,5000,2014,49
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,...,0,1987,0,98074,47.6168,-122.045,1800,7503,2015,28


In [10]:
df = pd.get_dummies(df, columns=['zipcode'], drop_first=True, dtype=int)

In [11]:
knn_features = ['sqft_living', 'bedrooms', 'bathrooms', 'floors', 
                'waterfront', 'view', 'condition', 'grade', 'yr_built', 'lat', 'long']
X = df[knn_features]
y = df['price']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# FIT on training data, TRANSFORM training data
X_train_scaled = scaler.fit_transform(X_train)

# ONLY TRANSFORM the test data (using the rules learned from train)
X_test_scaled = scaler.transform(X_test)

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

In [15]:
# ---------------------------------------------------------
# 1. Train the Baseline (Linear Regression)
# ---------------------------------------------------------
print("Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predict and evaluate
lr_preds = lr_model.predict(X_test_scaled)
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_r2 = r2_score(y_test, lr_preds)

print(f"Linear Regression - MAE: ${lr_mae:,.2f} | R2: {lr_r2:.4f}\n")

Training Linear Regression...
Linear Regression - MAE: $128,388.46 | R2: 0.6928



In [16]:
# ---------------------------------------------------------
# 2. Train the Heavyweight (Random Forest)
# ---------------------------------------------------------
print("Training Random Forest...")
# n_estimators=100 means 100 trees. n_jobs=-1 uses all your CPU cores to make it faster!
rf_model = RandomForestRegressor(n_estimators=300, max_depth=15,  random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# Predict and evaluate
rf_preds = rf_model.predict(X_test_scaled)
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_r2 = r2_score(y_test, rf_preds)

print(f"Random Forest - MAE: ${rf_mae:,.2f} | R2: {rf_r2:.4f}\n")

Training Random Forest...
Random Forest - MAE: $73,348.86 | R2: 0.8628



In [17]:
# ---------------------------------------------------------
# 3. Save the Best Model (For your FastAPI later)
# ---------------------------------------------------------
# We will save the Random Forest because it will have the better score
joblib.dump(rf_model, '../models/house_price_model.pkl', compress=3)

# CRITICAL: We also need to save the scaler! 
# When a user sends a new house, we must scale it the exact same way
joblib.dump(scaler, '../models/scaler.pkl', compress=3)

print("Models saved to disk as 'house_price_model.pkl' and 'scaler.pkl'!")

Models saved to disk as 'house_price_model.pkl' and 'scaler.pkl'!
